# Support Vector Classification (SVC): Titanic Survival Prediction

## Project objective
Build a reproducible binary classifier to predict whether a Titanic passenger survived, using a Support Vector Classifier (SVC).

This notebook covers:
- Data inspection and exploratory data analysis (EDA)
- Leakage-safe preprocessing with `ColumnTransformer` and `Pipeline`
- A baseline SVC and stratified cross-validation
- Hyperparameter tuning with `GridSearchCV`
- Final evaluation on a held-out test set
- Error analysis and model export

**Target:** `Survived` (`0` = did not survive, `1` = survived)

> The test set is reserved for final evaluation. Model selection and hyperparameter tuning use only the training set.

## 1. Imports and reproducibility

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
)
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    GridSearchCV,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC

import joblib

RANDOM_STATE = 42
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 100)

warnings.filterwarnings("default")

## 2. Load the dataset

Expected repository layout:

```text
project-root/
├── data/
│   └── titanicDataset.csv
├── notebooks/
│   └── svc.ipynb
└── models/
```

The path logic below supports running this notebook from either the project root or the `notebooks/` directory.

In [ ]:
candidate_paths = [
    Path("data/titanicDataset.csv"),
    Path("../data/titanicDataset.csv"),
]

DATA_PATH = next((path for path in candidate_paths if path.exists()), None)

if DATA_PATH is None:
    expected = "\n".join(f"- {path}" for path in candidate_paths)
    raise FileNotFoundError(
        "Could not find titanicDataset.csv. Check the dataset location. "
        f"Expected one of:\n{expected}"
    )

PROJECT_ROOT = DATA_PATH.resolve().parent.parent
df = pd.read_csv(DATA_PATH)

print(f"Dataset loaded from: {DATA_PATH.resolve()}")
print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")
display(df.head())

## 3. Initial data inspection

In [ ]:
display(df.info())
display(df.describe(include="all").T)

print("Duplicate rows:", df.duplicated().sum())
print("\nMissing values:")
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
})
display(missing_summary.sort_values("missing_percent", ascending=False))

print("\nTarget distribution:")
display(df["Survived"].value_counts().rename(index={0: "Did not survive", 1: "Survived"}))
display((df["Survived"].value_counts(normalize=True) * 100).round(2).rename(index={0: "Did not survive", 1: "Survived"}).rename("percent"))

## 4. Exploratory data analysis

These plots help explain the target distribution, missingness, and a few relationships with survival. They are descriptive; model settings are selected using cross-validation on the training data only.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

target_counts = df["Survived"].value_counts().reindex([0, 1], fill_value=0)
axes[0].bar(target_counts.index.astype(str), target_counts.values)
axes[0].set_title("Passenger survival count")
axes[0].set_xlabel("Survived (0 = No, 1 = Yes)")
axes[0].set_ylabel("Passengers")

missing_plot = missing_summary[missing_summary["missing_count"] > 0].sort_values("missing_percent")
axes[1].barh(missing_plot.index, missing_plot["missing_percent"])
axes[1].set_title("Missing values by feature")
axes[1].set_xlabel("Missing values (%)")
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

survival_by_sex = df.groupby("Sex", observed=False)["Survived"].mean().sort_index()
axes[0].bar(survival_by_sex.index.astype(str), survival_by_sex.values)
axes[0].set_title("Survival rate by sex")
axes[0].set_ylabel("Survival rate")
axes[0].set_ylim(0, 1)

survival_by_class = df.groupby("Pclass", observed=False)["Survived"].mean().sort_index()
axes[1].bar(survival_by_class.index.astype(str), survival_by_class.values)
axes[1].set_title("Survival rate by passenger class")
axes[1].set_xlabel("Passenger class")
axes[1].set_ylabel("Survival rate")
axes[1].set_ylim(0, 1)

for survived_value, label in [(0, "Did not survive"), (1, "Survived")]:
    ages = df.loc[df["Survived"] == survived_value, "Age"].dropna()
    axes[2].hist(ages, bins=25, alpha=0.6, label=label)
axes[2].set_title("Age distribution by survival")
axes[2].set_xlabel("Age")
axes[2].set_ylabel("Passengers")
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
numeric_columns_for_corr = [
    column for column in ["Survived", "Pclass", "Age", "SibSp", "Parch", "Fare"]
    if column in df.columns
]

plt.figure(figsize=(8, 5))
sns.heatmap(
    df[numeric_columns_for_corr].corr(),
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    vmin=-1,
    vmax=1,
)
plt.title("Correlation among selected numeric features")
plt.tight_layout()
plt.show()

## 5. Define the target and features

The model uses `Pclass`, `Sex`, `Age`, `SibSp`, `Parch`, `Fare`, and `Embarked`.

`PassengerId` is an identifier. `Name`, `Ticket`, and `Cabin` are excluded from this baseline because they require additional feature engineering and/or have high cardinality or substantial missingness. They can be explored later as a separate experiment.

The preprocessing steps are placed inside a pipeline so imputers, encoders, and scalers are fitted only on the training folds during cross-validation.

In [ ]:
required_columns = [
    "Survived", "Pclass", "Sex", "Age", "SibSp",
    "Parch", "Fare", "Embarked"
]
missing_columns = sorted(set(required_columns) - set(df.columns))
if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {missing_columns}")

# This notebook assumes the target is labeled 0/1 and known for every row.
df = df.dropna(subset=["Survived"]).copy()
if not set(df["Survived"].unique()).issubset({0, 1}):
    raise ValueError("Expected Survived to contain only 0 and 1.")

FEATURES = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]
TARGET = "Survived"

X = df[FEATURES].copy()
y = df[TARGET].astype(int).copy()

numerical_features = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
categorical_features = ["Sex", "Embarked"]

print("Feature columns:", FEATURES)
print("Target counts:")
display(y.value_counts().sort_index().rename(index={0: "Did not survive", 1: "Survived"}))

## 6. Train-test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Training rows: {len(X_train)}")
print(f"Test rows: {len(X_test)}")
print("\nTraining target proportions:")
display(y_train.value_counts(normalize=True).sort_index().rename(index={0: "Did not survive", 1: "Survived"}))

## 7. Build the preprocessing and SVC pipeline

- Numerical features: median imputation followed by standardization.
- Categorical features: most-frequent imputation followed by one-hot encoding.
- Classifier: RBF-kernel SVC.

Scaling is important for SVMs because their decision boundary depends on distances and dot products.

In [ ]:
numerical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numerical_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features),
])

svc_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", SVC(kernel="rbf")),
])

svc_pipeline

## 8. Establish a baseline with cross-validation

Before tuning, measure the default SVC using stratified 5-fold cross-validation on the **training data only**. F1-score is used as the selection metric for the positive class (`Survived = 1`).

This result is a baseline; it is not the final test-set score.

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

baseline_cv_scores = cross_val_score(
    svc_pipeline,
    X_train,
    y_train,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
)

print("Default SVC CV F1 scores:", np.round(baseline_cv_scores, 4))
print(f"Mean CV F1: {baseline_cv_scores.mean():.4f}")
print(f"Standard deviation: {baseline_cv_scores.std():.4f}")

## 9. Hyperparameter tuning with GridSearchCV

The grid searches over:
- `C`: regularization strength (smaller values apply stronger regularization).
- `gamma`: influence range of individual training examples for the RBF kernel.
- `class_weight`: optionally compensates for class imbalance.

The cross-validation strategy is shuffled and stratified. The held-out test set is not used during model selection.

In [ ]:
param_grid = {
    "classifier__C": [0.1, 1, 10, 100],
    "classifier__gamma": ["scale", 0.01, 0.1, 1],
    "classifier__class_weight": [None, "balanced"],
}

grid_search = GridSearchCV(
    estimator=svc_pipeline,
    param_grid=param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    refit=True,
    return_train_score=True,
)

grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_

print("Best parameters:")
print(grid_search.best_params_)
print(f"\nBest mean CV F1: {grid_search.best_score_:.4f}")

In [ ]:
cv_results = (
    pd.DataFrame(grid_search.cv_results_)
    .sort_values("rank_test_score")
    [[
        "rank_test_score",
        "mean_test_score",
        "std_test_score",
        "mean_train_score",
        "params",
    ]]
    .head(10)
    .reset_index(drop=True)
)

display(cv_results)

## 10. Final evaluation on the held-out test set

This is the final evaluation of the selected model. Avoid repeatedly changing model settings in response to these results; doing so would turn the test set into part of the tuning process.

In [ ]:
y_pred = best_model.predict(X_test)

# SVC has decision_function scores even with probability=False.
y_score = best_model.decision_function(X_test)

final_metrics = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Balanced accuracy",
        "Precision (survived)",
        "Recall (survived)",
        "F1-score (survived)",
        "ROC-AUC",
    ],
    "Score": [
        accuracy_score(y_test, y_pred),
        balanced_accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred, zero_division=0),
        recall_score(y_test, y_pred, zero_division=0),
        f1_score(y_test, y_pred, zero_division=0),
        roc_auc_score(y_test, y_score),
    ],
})

display(final_metrics.style.format({"Score": "{:.4f}"}))

print(classification_report(
    y_test,
    y_pred,
    labels=[0, 1],
    target_names=["Did not survive", "Survived"],
    zero_division=0,
))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ConfusionMatrixDisplay.from_estimator(
    best_model,
    X_test,
    y_test,
    display_labels=["Did not survive", "Survived"],
    cmap="Blues",
    ax=axes[0],
)
axes[0].set_title("Confusion matrix")

RocCurveDisplay.from_estimator(
    best_model,
    X_test,
    y_test,
    ax=axes[1],
)
axes[1].set_title("ROC curve")

plt.tight_layout()
plt.show()

## 11. Compare against a simple baseline

A model should be compared against a simple reference model. The `DummyClassifier` below always predicts the most frequent class observed in the training set. Both it and the selected SVC are evaluated on the same held-out test examples.

In [ ]:
dummy_model = DummyClassifier(strategy="most_frequent")
dummy_model.fit(X_train, y_train)
dummy_pred = dummy_model.predict(X_test)

comparison = pd.DataFrame([
    {
        "Model": "DummyClassifier (most frequent)",
        "Accuracy": accuracy_score(y_test, dummy_pred),
        "Balanced accuracy": balanced_accuracy_score(y_test, dummy_pred),
        "Precision (survived)": precision_score(y_test, dummy_pred, zero_division=0),
        "Recall (survived)": recall_score(y_test, dummy_pred, zero_division=0),
        "F1 (survived)": f1_score(y_test, dummy_pred, zero_division=0),
    },
    {
        "Model": "Tuned SVC",
        "Accuracy": accuracy_score(y_test, y_pred),
        "Balanced accuracy": balanced_accuracy_score(y_test, y_pred),
        "Precision (survived)": precision_score(y_test, y_pred, zero_division=0),
        "Recall (survived)": recall_score(y_test, y_pred, zero_division=0),
        "F1 (survived)": f1_score(y_test, y_pred, zero_division=0),
    },
])

display(comparison.set_index("Model").style.format("{:.4f}"))

## 12. Inspect misclassified examples

This section shows test-set rows the selected model classified incorrectly. Use it to generate hypotheses for further experiments, not to claim that individual errors have a single known cause.

In [ ]:
misclassified_mask = (y_test.to_numpy() != y_pred)
misclassified_indices = y_test.index[misclassified_mask]

error_columns = [
    "Survived", "Pclass", "Sex", "Age", "SibSp",
    "Parch", "Fare", "Embarked",
]
misclassified = df.loc[misclassified_indices, error_columns].copy()
misclassified["PredictedSurvived"] = y_pred[misclassified_mask]

print(f"Misclassified test examples: {len(misclassified)} of {len(y_test)}")
display(misclassified.head(20))

## 13. Save the complete fitted pipeline

Saving the complete pipeline preserves the fitted imputer, scaler, encoder, and classifier. At prediction time, pass a DataFrame containing the same raw feature columns used during training.

Only load pickle/joblib model files from trusted sources. Scikit-learn model objects may not be compatible across arbitrary library versions.

In [ ]:
model_dir = PROJECT_ROOT / "models"
model_dir.mkdir(parents=True, exist_ok=True)

model_path = model_dir / "svc_titanic_model.joblib"
joblib.dump(best_model, model_path)

print(f"Saved model to: {model_path.resolve()}")

In [ ]:
# Example: load the model and predict for a few held-out passengers.
loaded_model = joblib.load(model_path)

sample_passengers = X_test.head(5)
sample_predictions = loaded_model.predict(sample_passengers)

display(sample_passengers.assign(PredictedSurvived=sample_predictions))

## 14. Conclusions and limitations

Complete these notes after running the notebook:

1. **Best cross-validation F1-score:** record `grid_search.best_score_`.
2. **Test F1-score and recall for survivors:** record the values from the final evaluation table.
3. **Baseline comparison:** state whether tuned SVC improves on `DummyClassifier`, and on which metrics.
4. **Limitations:** this is a small, historical dataset; results do not establish generalization to other populations or modern passenger data.
5. **Further experiments:** compare engineered `Title` and `FamilySize` features, other SVM kernels, and alternative models using the same cross-validation procedure.

Do not claim the model is production-ready based solely on this exercise.